# 1 · Brunel-based recurrent spiking network

A homogeneous LIF network with fixed excitatory/inhibitory in-degree. Default external drive is constant, matching the supplied Dummer notebooks. Voltages are relative to a zero leak reversal. The quick preset is for development, not a completed paper reproduction.

## Parameters
Edit `config.json` once for all three notebooks. `g` is inhibitory/excitatory weight ratio; `eta` is used only in Poisson-drive mode. All times in the JSON are seconds.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sc_cortex.config import Config
from sc_cortex.results import save_result, load_result

c = Config.load('config.json')
print(c)
print('Run ID:', c.fingerprint)


## Simulate
Euler updates with delta voltage-jump synapses. Record a small E/I sample after burn-in; retain the full population rate to inspect collective activity. Sparse fixed in-degree connections permit autapses.

In [ ]:
from sc_cortex.models import run_network
network = run_network(c)
path = save_result('network', c, network)
print('Saved:', path)
print(f"Sample rate: {network['mean_rate']:.2f} Hz")
print(f"Full population rate: {network['population_mean_rate']:.2f} Hz")

## Inspect activity and individual-neuron spectra
The averaged single-neuron PSD is different from the PSD of the population rate. We plot the two-sided spectral convention at positive frequencies: a Poisson spectrum approaches its firing rate, not twice its rate.

In [ ]:
from scipy.ndimage import gaussian_filter1d
fig, ax = plt.subplots(3, 1, figsize=(10, 9), constrained_layout=True)
mask = network['spike_t'] < .3

ax[0].plot(network['spike_t'][mask],network['spike_i'][mask], '|', color='#d16b91', ms=2)

ax[0].set(xlabel='Time after burn-in (s)', ylabel='Recorded neuron', title='Sample raster')
ax[1].plot(network['population_t'], gaussian_filter1d(network['population_rate'], .002/c.dt), color='#dc8b45')
ax[1].set(xlabel='Time after burn-in (s)', ylabel='Population rate (Hz)', title='2-ms Gaussian smoothing for display only')
keep = (network['freq'] > 0) & (network['freq'] <= 300)
ax[2].semilogy(network['freq'][keep], network['psd'][keep], color='#d16b91')
ax[2].set(xlabel='Frequency (Hz)', ylabel='Single-neuron PSD (Hz)', title='Mean over recorded neurons')
plt.show()